In [19]:
# ============================================================
# CONFIG
# ============================================================

import numpy as np
import pandas as pd
import GEOparse

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

TRAIN_GSE = "GSE17705"
TEST_GSE  = "GSE55348"

RANDOM_STATE = 42
K_VALUES = [20, 50, 100, 500]

print(f"\nTRAIN: {TRAIN_GSE}")
print(f"TEST: {TEST_GSE}")


# ============================================================
# FUNCIONES
# ============================================================

def is_her2_positive(metadata):
    chars = metadata.get("characteristics_ch1", [])

    for c in chars:
        c = c.lower()

        if (
            "her2+" in c or
            "her2 positive" in c or
            "her2: positive" in c or
            ("erbb2" in c and ("amplified" in c or "+" in c))
        ):
            return True

    return False


def extract_label(metadata):
    chars = metadata.get("characteristics_ch1", [])

    for c in chars:
        c = c.lower()

        if "event:" in c or "metastasis" in c or "distant relapse" in c:
            try:
                return int(c.split(":")[-1].strip())
            except:
                pass

    return None


# ============================================================
# A: CARGA TRAIN
# ============================================================

print(f"\n=== CARGANDO TRAIN: {TRAIN_GSE} ===")
gse = GEOparse.get_GEO(TRAIN_GSE, destdir="../data")

# 🔍 DEBUG metadata
print("\n=== DEBUG METADATA TRAIN ===")
for i, (gsm_name, gsm) in enumerate(gse.gsms.items()):
    print(gsm.metadata.get("characteristics_ch1", []))
    if i > 10:
        break


# ============================================================
# B: CONSTRUCCIÓN TRAIN (HER2 si existe)
# ============================================================

expr_data = {}
y = []

her2_count = 0

for gsm_name, gsm in gse.gsms.items():

    if is_her2_positive(gsm.metadata):
        her2_count += 1

# 👉 decidir si usamos HER2 o no
use_her2 = her2_count > 10

print(f"\nHER2+ samples detectadas: {her2_count}")
print("Usando filtro HER2:", use_her2)


for gsm_name, gsm in gse.gsms.items():

    if use_her2 and not is_her2_positive(gsm.metadata):
        continue

    label = extract_label(gsm.metadata)
    if label is None:
        continue

    expr_data[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]
    y.append(label)

X = pd.DataFrame(expr_data).T
X_log = np.log2(X + 1)
y = np.array(y)

print("TRAIN shape:", X.shape)
print("Distribución clases:", np.bincount(y))


# ============================================================
# C: MAPPING TRAIN
# ============================================================

platform = gse.gpls[list(gse.gpls.keys())[0]]
annot = platform.table

gene_col = "Gene Symbol" if "Gene Symbol" in annot.columns else "Symbol"
print("Columna genes TRAIN:", gene_col)

map_train = dict(zip(annot['ID'], annot[gene_col]))

X_train_genes = X_log.rename(columns=map_train)
X_train_genes = X_train_genes.T.groupby(level=0).mean().T


# ============================================================
# D: CARGA TEST
# ============================================================

print(f"\n=== CARGANDO TEST: {TEST_GSE} ===")
gse_test = GEOparse.get_GEO(TEST_GSE, destdir="../data")

expr_data_test = {}
y_test = []

her2_count_test = 0

for gsm_name, gsm in gse_test.gsms.items():
    if is_her2_positive(gsm.metadata):
        her2_count_test += 1

use_her2_test = her2_count_test > 5

print(f"HER2+ test detectadas: {her2_count_test}")
print("Usando HER2 en test:", use_her2_test)


for gsm_name, gsm in gse_test.gsms.items():

    if use_her2_test and not is_her2_positive(gsm.metadata):
        continue

    label = extract_label(gsm.metadata)
    if label is None:
        continue

    expr_data_test[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]
    y_test.append(label)

X_test = pd.DataFrame(expr_data_test).T
X_test_log = np.log2(X_test + 1)
y_test = np.array(y_test)

print("TEST shape:", X_test.shape)
print("Distribución test:", np.bincount(y_test))


# ============================================================
# E: MAPPING TEST
# ============================================================

platform_test = gse_test.gpls[list(gse_test.gpls.keys())[0]]
annot_test = platform_test.table

gene_col_test = "Gene Symbol" if "Gene Symbol" in annot_test.columns else "Symbol"
print("Columna genes TEST:", gene_col_test)

map_test = dict(zip(annot_test['ID'], annot_test[gene_col_test]))

X_test_genes = X_test_log.rename(columns=map_test)
X_test_genes = X_test_genes.T.groupby(level=0).mean().T


# ============================================================
# F: INTERSECCIÓN
# ============================================================

common_genes = list(set(X_train_genes.columns) & set(X_test_genes.columns))

print("Genes comunes:", len(common_genes))

X_train_final = X_train_genes[common_genes].copy()
X_test_final  = X_test_genes[common_genes].copy()


# ============================================================
# G: CV
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for K in K_VALUES:

    pipe = Pipeline([
        ("variance", VarianceThreshold(0.01)),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=K)),
        ("model", LogisticRegression(max_iter=1000))
    ])

    scores = cross_validate(
        pipe,
        X_train_final,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    print(f"K={K} → AUC CV: {np.mean(scores['test_score']):.3f}")


# ============================================================
# H: MODELO FINAL (L1)
# ============================================================

l1_pipe = Pipeline([
    ("variance", VarianceThreshold(0.01)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.1,
        max_iter=5000
    ))
])

l1_pipe.fit(X_train_final, y)

coefs = l1_pipe.named_steps["model"].coef_[0]
mask = l1_pipe.named_steps["variance"].get_support()

genes_after_variance = X_train_final.columns[mask]
selected_genes = genes_after_variance[coefs != 0]

print("\nGenes seleccionados:", len(selected_genes))


# ============================================================
# I: VALIDACIÓN EXTERNA
# ============================================================

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train_final[selected_genes], y)

y_pred = model.predict_proba(X_test_final[selected_genes])[:, 1]

auc = roc_auc_score(y_test, y_pred)
auc_inv = roc_auc_score(y_test, 1 - y_pred)

print("\n=== RESULTADOS ===")
print("AUC:", auc)
print("AUC invertido:", auc_inv)

07-Apr-2026 22:03:35 DEBUG utils - Directory ../data already exists. Skipping.
07-Apr-2026 22:03:35 INFO GEOparse - File already exist: using local version.
07-Apr-2026 22:03:35 INFO GEOparse - Parsing ../data\GSE17705_family.soft.gz: 
07-Apr-2026 22:03:35 DEBUG GEOparse - DATABASE: GeoMiame
07-Apr-2026 22:03:35 DEBUG GEOparse - SERIES: GSE17705
07-Apr-2026 22:03:35 DEBUG GEOparse - PLATFORM: GPL96



TRAIN: GSE17705
TEST: GSE55348

=== CARGANDO TRAIN: GSE17705 ===


07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441624
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441625
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441626
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441627
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441628
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441629
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441630
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441631
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441632
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441633
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441634
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441635
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441636
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441637
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441638
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441639
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GSM441640
07-Apr-2026 22:03:36 DEBUG GEOparse - SAMPLE: GS


=== DEBUG METADATA TRAIN ===
['patient id: 1070', 'profiling lab: JBI', 'tissue: breast cancer', 'estrogen receptor (er) status: ER+', 'endocrine therapy: Tamoxifen treatment for 5 years', 'nodal status (0=negative, 1=positive, na=not applicable): 0', 'distant relapse (1=dr, 0 censored): 0', 'event time (years): 7.578082192']
['patient id: 1089', 'profiling lab: JBI', 'tissue: breast cancer', 'estrogen receptor (er) status: ER+', 'endocrine therapy: Tamoxifen treatment for 5 years', 'nodal status (0=negative, 1=positive, na=not applicable): 0', 'distant relapse (1=dr, 0 censored): 0', 'event time (years): 2.676712329']
['patient id: 1097', 'profiling lab: JBI', 'tissue: breast cancer', 'estrogen receptor (er) status: ER+', 'endocrine therapy: Tamoxifen treatment for 5 years', 'nodal status (0=negative, 1=positive, na=not applicable): 0', 'distant relapse (1=dr, 0 censored): 0', 'event time (years): 7.739726027']
['patient id: 1103', 'profiling lab: JBI', 'tissue: breast cancer', 'estr

C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)
07-Apr-2026 22:03:48 DEBUG utils - Directory ../data already exists. Skipping.
07-Apr-2026 22:03:48 INFO GEOparse - File already exist: using local version.
07-Apr-2026 22:03:48 INFO GEOparse - Parsing ../data\GSE55348_family.soft.gz: 
07-Apr-2026 22:03:48 DEBUG GEOparse - DATABASE: GeoMiame
07-Apr-2026 22:03:48 DEBUG GEOparse - SERIES: GSE55348
07-Apr-2026 22:03:48 DEBUG GEOparse - PLATFORM: GPL14951


TRAIN shape: (298, 22283)
Distribución clases: [227  71]
Columna genes TRAIN: Gene Symbol

=== CARGANDO TEST: GSE55348 ===


07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334487
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334488
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334489
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334490
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334491
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334492
07-Apr-2026 22:03:49 DEBUG GEOparse - SAMPLE: GSM1334493
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334494
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334495
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334496
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334497
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334498
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334499
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334500
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334501
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334502
07-Apr-2026 22:03:50 DEBUG GEOparse - SAMPLE: GSM1334503
07-Apr-2026 22:03:50 DEBUG GEOp

HER2+ test detectadas: 0
Usando HER2 en test: False
TEST shape: (53, 29377)
Distribución test: [30 23]
Columna genes TEST: Symbol
Genes comunes: 11312
K=20 → AUC CV: 0.654
K=50 → AUC CV: 0.620
K=100 → AUC CV: 0.580
K=500 → AUC CV: 0.668

Genes seleccionados: 87

=== RESULTADOS ===
AUC: 0.3521739130434782
AUC invertido: 0.5


In [8]:
# ============================================================
# A: CARGA TRAIN
# ============================================================

print(f"\n=== CARGANDO TRAIN: {TRAIN_GSE} ===")

gse = GEOparse.get_GEO(TRAIN_GSE, destdir="../data")

expr_data = {}
for gsm_name, gsm in gse.gsms.items():
    expr_data[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]

X = pd.DataFrame(expr_data).T
X_log = np.log2(X + 1)

print("Shape X:", X.shape)
print("NaNs en X:", np.isnan(X_log).sum().sum())


# ============================================================
# LABELS TRAIN
# ============================================================

y = []

for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    label = None

    for c in chars:
        c = c.lower()

        if "event:" in c or "metastasis" in c or "distant relapse" in c:
            try:
                label = int(c.split(":")[-1].strip())
            except:
                pass

    if label is None:
        raise ValueError(f"No label en {gsm_name}")

    y.append(label)

y = np.array(y)
print("Distribución clases:", np.bincount(y))


# ============================================================
# B: MAPPING TRAIN → GENES
# ============================================================

platform = gse.gpls[list(gse.gpls.keys())[0]]
annot = platform.table

gene_col = "Gene Symbol" if "Gene Symbol" in annot.columns else "Symbol"
print("Columna genes TRAIN:", gene_col)

map_train = dict(zip(annot['ID'], annot[gene_col]))

X_train_genes = X_log.rename(columns=map_train)
X_train_genes = X_train_genes.T.groupby(level=0).mean().T


# ============================================================
# C: CARGA TEST
# ============================================================

print(f"\n=== CARGANDO TEST: {TEST_GSE} ===")

gse_test = GEOparse.get_GEO(TEST_GSE, destdir="../data")

expr_data_test = {}
for gsm_name, gsm in gse_test.gsms.items():
    expr_data_test[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]

X_test = pd.DataFrame(expr_data_test).T
X_test_log = np.log2(X_test + 1)


# ============================================================
# LABELS TEST
# ============================================================

y_test = []

for gsm_name, gsm in gse_test.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    label = None

    for c in chars:
        c = c.lower()

        if "event:" in c or "metastasis" in c or "distant relapse" in c:
            try:
                label = int(c.split(":")[-1].strip())
            except:
                pass

    if label is None:
        raise ValueError(f"No label en {gsm_name}")

    y_test.append(label)

y_test = np.array(y_test)


# ============================================================
# D: MAPPING TEST → GENES
# ============================================================

platform_test = gse_test.gpls[list(gse_test.gpls.keys())[0]]
annot_test = platform_test.table

gene_col_test = "Gene Symbol" if "Gene Symbol" in annot_test.columns else "Symbol"
print("Columna genes TEST:", gene_col_test)

map_test = dict(zip(annot_test['ID'], annot_test[gene_col_test]))

X_test_genes = X_test_log.rename(columns=map_test)
X_test_genes = X_test_genes.T.groupby(level=0).mean().T


# ============================================================
# E: INTERSECCIÓN DE GENES (CLAVE)
# ============================================================

common_genes = list(set(X_train_genes.columns) & set(X_test_genes.columns))

print("Genes comunes:", len(common_genes))

X_train_final = X_train_genes[common_genes].copy()
X_test_final  = X_test_genes[common_genes].copy()


# ============================================================
# F: EXPERIMENTACIÓN (CV REALISTA)
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

all_results = []

for K in K_VALUES:

    pipelines = {

        "LR baseline": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),

        "LR + ANOVA": Pipeline([
            ("variance", VarianceThreshold(0.01)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("select", SelectKBest(f_classif, k=K)),
            ("model", LogisticRegression(max_iter=1000))
        ])
    }

    for name, pipe in pipelines.items():

        scores = cross_validate(
            pipe,
            X_train_final,
            y,
            cv=cv,
            scoring={"roc_auc": "roc_auc", "f1": "f1"},
            n_jobs=2
        )

        all_results.append({
            "Model": name,
            "K": K,
            "ROC-AUC": np.mean(scores["test_roc_auc"]),
            "F1": np.mean(scores["test_f1"])
        })


# ============================================================
# G: MODELO FINAL (L1)
# ============================================================

l1_pipe = Pipeline([
    ("variance", VarianceThreshold(0.01)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.1,
        max_iter=5000
    ))
])

l1_pipe.fit(X_train_final, y)

coefs = l1_pipe.named_steps["model"].coef_[0]
mask = l1_pipe.named_steps["variance"].get_support()

genes_after_variance = X_train_final.columns[mask]
selected_genes = genes_after_variance[coefs != 0]

print("\nGenes seleccionados:", len(selected_genes))


# ============================================================
# H: VALIDACIÓN EXTERNA
# ============================================================

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train_final[selected_genes], y)

y_pred = model.predict_proba(X_test_final[selected_genes])[:, 1]

auc = roc_auc_score(y_test, y_pred)
auc_inv = roc_auc_score(y_test, 1 - y_pred)

print("\n=== RESULTADOS ===")
print("AUC:", auc)
print("AUC invertido:", auc_inv)


# ============================================================
# I: RESULTADOS CV
# ============================================================

results_df = pd.DataFrame(all_results).sort_values(by="ROC-AUC", ascending=False)

print("\n=== RESULTADOS CV ===")
print(results_df)


=== INSPECCIÓN DE METADATA ===

Muestra: GSM441624

[characteristics_ch1]
- patient id: 1070
- profiling lab: JBI
- tissue: breast cancer
- estrogen receptor (er) status: ER+
- endocrine therapy: Tamoxifen treatment for 5 years
- nodal status (0=negative, 1=positive, na=not applicable): 0
- distant relapse (1=dr, 0 censored): 0
- event time (years): 7.578082192

[OTRAS METADATA CLAVES]

title:
- Tissue_BC_Tamoxifen_1070

description:
- 1070.CEL

source_name_ch1:
- tissue from breast tumor excision

Muestra: GSM441625

[characteristics_ch1]
- patient id: 1089
- profiling lab: JBI
- tissue: breast cancer
- estrogen receptor (er) status: ER+
- endocrine therapy: Tamoxifen treatment for 5 years
- nodal status (0=negative, 1=positive, na=not applicable): 0
- distant relapse (1=dr, 0 censored): 0
- event time (years): 2.676712329

[OTRAS METADATA CLAVES]

title:
- Tissue_BC_Tamoxifen_1089

description:
- 1089.CEL

source_name_ch1:
- tissue from breast tumor excision

Muestra: GSM441626

[ch

In [ ]:

# ============================================================
# D: VALIDACIÓN EXTERNA (CORREGIDO Y ROBUSTO)
# ============================================================

print("\n=== VALIDACIÓN EXTERNA ===")

gse_58984 = GEOparse.get_GEO("GSE58984", destdir="../data")

# ------------------------------------------------------------
# D1. Cargar datos
# ------------------------------------------------------------

expr_data_58984 = {}
for gsm_name, gsm in gse_58984.gsms.items():
    expr_data_58984[gsm_name] = gsm.table.set_index("ID_REF")["VALUE"]

X_58984 = pd.DataFrame(expr_data_58984).T
X_58984_log = np.log2(X_58984 + 1)

# ------------------------------------------------------------
# D2. Labels
# ------------------------------------------------------------

y_58984 = []
for gsm_name in X_58984.index:
    gsm = gse_58984.gsms[gsm_name]
    chars = gsm.metadata.get("characteristics_ch1", [])

    for c in chars:
        if "distant metastasis" in c.lower():
            y_58984.append(int(c.split(":")[-1].strip()))

y_58984 = np.array(y_58984)

print("Distribución externa:", np.bincount(y_58984))


# ------------------------------------------------------------
# D3. Mapping a genes (igual que antes)
# ------------------------------------------------------------

print("\n=== MAPEO A GENES (GSE58984) ===")

platform_58984 = gse_58984.gpls[list(gse_58984.gpls.keys())[0]]
annot_58984 = platform_58984.table

map_58984 = dict(zip(annot_58984['ID'], annot_58984['Gene Symbol']))

X_58984_genes = X_58984_log.rename(columns=map_58984)
# Limpiar nombres
X_58984_genes.columns = [
    str(g).split("///")[0].strip() if g is not None else None
    for g in X_58984_genes.columns
]

# Eliminar columnas sin nombre
X_58984_genes = X_58984_genes.loc[:, X_58984_genes.columns.notna()]

# AGRUPAR genes duplicados (igual que en train)
X_58984_genes = X_58984_genes.T.groupby(level=0).mean().T
# LIMPIEZA de nombres (CLAVE)
X_58984_genes.columns = [
    str(g).split("///")[0].strip() if g is not None else None
    for g in X_58984_genes.columns
]

# Eliminamos columnas sin nombre válido
X_58984_genes = X_58984_genes.loc[:, X_58984_genes.columns.notna()]


# ------------------------------------------------------------
# D4. También limpiamos los genes del training
# ------------------------------------------------------------

mapped_selected_genes_clean = [
    g.split("///")[0].strip() if isinstance(g, str) else g
    for g in mapped_selected_genes
]


# ------------------------------------------------------------
# D5. Intersección REAL
# ------------------------------------------------------------

common_genes = [
    g for g in mapped_selected_genes_clean
    if g in X_58984_genes.columns
]

print("Genes comunes:", len(common_genes))
print(common_genes)


# ------------------------------------------------------------
# D6. CONTROL (MUY IMPORTANTE)
# ------------------------------------------------------------

if len(common_genes) == 0:
    raise ValueError(
        "No hay genes comunes entre datasets. "
        "Esto probablemente se debe a diferencias de anotación entre plataformas."
    )


# ------------------------------------------------------------
# D7. Entrenamiento y evaluación
# ------------------------------------------------------------

# 1. Nos quedamos SOLO con los probes seleccionados
X_train = X_log[selected_genes].copy()

# 2. Mapping probe → gen
probe_to_gene = dict(zip(selected_genes, mapped_selected_genes_clean))

# 3. Renombrar
X_train = X_train.rename(columns=probe_to_gene)

# 4. Agrupar genes duplicados
X_train = X_train.T.groupby(level=0).mean().T

# 5. Seleccionar genes comunes
X_train = X_train[common_genes]

# ------------------------------------------------------------
# IMPORTANTE: asegurar mismo orden en test
# ------------------------------------------------------------

# Nos aseguramos de que test tiene SOLO esos genes
X_test = X_58984_genes.copy()

# Nos quedamos con los genes comunes (y en mismo orden)
X_test = X_test.reindex(columns=common_genes)

# ------------------------------------------------------------
# DEBUG
# ------------------------------------------------------------

print("Train columns:", list(X_train.columns))
print("Test columns:", list(X_test.columns))

print("Shape train:", X_train.shape)
print("Shape test:", X_test.shape)

# ------------------------------------------------------------
# Modelo
# ------------------------------------------------------------

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

final_model.fit(X_train, y)

# Predicción probabilística
y_pred = final_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------
# MÉTRICAS
# ------------------------------------------------------------

from sklearn.metrics import confusion_matrix

# AUC
auc = roc_auc_score(y_58984, y_pred)
auc_inv = roc_auc_score(y_58984, 1 - y_pred)

# Convertimos probabilidades a clases (umbral 0.5)
y_pred_class = (y_pred >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_58984, y_pred_class).ravel()

sensibilidad = tp / (tp + fn) if (tp + fn) > 0 else 0
especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n=== MÉTRICAS EXTERNAS ===")
print("AUC:", auc)
print("AUC invertido:", auc_inv)
print("Sensibilidad:", sensibilidad)
print("Especificidad:", especificidad)

